# UK Online Retail Sales Analysis — Data Cleaning & Preparation

This notebook loads, inspects, cleans, and exports the Online Retail II dataset
(UCI ML Repository), preparing it for analysis in PostgreSQL.

**Dataset:** Real transactional data from a UK-based online gift retailer,
covering December 2009 – December 2011, ~1 million rows across two sheets.

**Workflow:**
1. Load raw data
2. Explore & assess data quality
3. Clean the data (with documented reasoning for each decision)
4. Export to CSV
5. Load directly into PostgreSQL


## 1. Load the Raw Data

In [ ]:
import pandas as pd

# The dataset ships as a single Excel file with two sheets, one per year
df_2009 = pd.read_excel("~/Downloads/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010 = pd.read_excel("~/Downloads/online_retail_II.xlsx", sheet_name="Year 2010-2011")

# Combine both years into a single dataframe
df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f"Total rows loaded: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
df.head()


## 2. Explore & Assess Data Quality

Before cleaning anything, check the structure, missing values, duplicates, and known problem areas (cancellations, negative quantities/prices) so cleaning decisions are evidence-based rather than assumed.

In [ ]:
# Basic structure
print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)

# Missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Exact duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Cancelled orders are recorded as invoices starting with 'C'
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print("\nCancelled order rows:", cancelled.shape[0])

# Negative/zero quantities (largely overlaps with cancellations)
print("\nNegative or zero Quantity rows:", (df['Quantity'] <= 0).sum())

# Negative/zero prices (likely data entry errors, not real sales)
print("\nNegative or zero Price rows:", (df['Price'] <= 0).sum())

# Country distribution
print("\nUnique countries:", df['Country'].nunique())
print(df['Country'].value_counts().head(10))


### Data Quality Summary & Cleaning Decisions

| Issue | Count | Decision | Reasoning |
|---|---|---|---|
| Missing `Customer ID` | 243,007 | Remove | Cannot do customer-level analysis (segmentation, retention, RFM) without it |
| Missing `Description` | 4,382 | Remove | Small proportion; cannot analyse unnamed products |
| Duplicate rows | 34,335 | Remove | Likely system logging errors, not genuine repeat transactions |
| Cancelled orders (Invoice starts with 'C') | 19,494 | **Keep, but separate** | Cancellations are a valid business signal (returns/refunds) — moved to their own table rather than deleted |
| Negative Quantity | 22,950 | Removed via cancellation split | This is *how* cancellations are recorded (negative quantity = returned items) |
| Negative/zero Price | 6,207 | Remove | Likely data entry errors or write-offs, not genuine sales transactions |

This approach preserves cancellation data for return-rate analysis rather than
discarding it outright, while ensuring the core sales table only contains
valid, completed, attributable transactions.

## 3. Clean the Data

In [ ]:
# Keep an untouched copy of the raw data for reference/reproducibility
df_raw = df.copy()

# Separate cancellations into their own dataset before cleaning —
# these are valid business data (returns), not errors
df_cancellations = df[df['Invoice'].astype(str).str.startswith('C')].copy()

# Build the clean sales dataset: completed, valid, attributable transactions only
df_clean = df[
    (~df['Invoice'].astype(str).str.startswith('C')) &   # exclude cancellations (handled separately)
    (df['Quantity'] > 0) &                                 # exclude negative/zero quantities
    (df['Price'] > 0) &                                    # exclude negative/zero prices
    (df['Customer ID'].notnull()) &                        # exclude rows with no customer attribution
    (df['Description'].notnull())                          # exclude rows with no product description
].copy()

# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Add a calculated TotalPrice column — needed for revenue analysis
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']

# Customer ID loads as float (due to NaNs in the original data) — safe to convert to int now
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

print(f"Original rows:                {df.shape[0]:,}")
print(f"Clean rows:                   {df_clean.shape[0]:,}")
print(f"Cancellation rows (separate): {df_cancellations.shape[0]:,}")
print(f"Retention rate:               {df_clean.shape[0] / df.shape[0] * 100:.1f}%")

df_clean.head()


## 4. Prepare Columns for PostgreSQL Import

Rename columns to lowercase snake_case to exactly match the target PostgreSQL table schema, avoiding import-mapping issues.

In [ ]:
column_mapping = {
    'Invoice': 'invoice',
    'StockCode': 'stock_code',
    'Description': 'description',
    'Quantity': 'quantity',
    'InvoiceDate': 'invoice_date',
    'Price': 'price',
    'Customer ID': 'customer_id',
    'Country': 'country',
    'TotalPrice': 'total_price'
}

df_clean_export = df_clean.rename(columns=column_mapping)

# Cancellations table has no TotalPrice column
cancellation_mapping = {k: v for k, v in column_mapping.items() if k != 'TotalPrice'}
df_cancellations_export = df_cancellations.rename(columns=cancellation_mapping)

print("clean_sales columns:", df_clean_export.columns.tolist())
print("cancellations columns:", df_cancellations_export.columns.tolist())


## 5. Export to CSV

(Optional checkpoint — useful for backups or if importing via a GUI tool like DBeaver.)

In [ ]:
df_clean_export.to_csv("clean_sales.csv", index=False)
df_cancellations_export.to_csv("cancellations.csv", index=False)

print("Files exported:")
print(f"  clean_sales.csv    ({df_clean_export.shape[0]:,} rows)")
print(f"  cancellations.csv  ({df_cancellations_export.shape[0]:,} rows)")


## 6. Load Directly into PostgreSQL

Loading directly via SQLAlchemy is more reliable than a GUI import wizard
(avoids column-mapping issues) and better reflects a real analyst workflow.

**Prerequisite (run once):** the `clean_sales` and `cancellations` tables
must already exist in the `retail_analysis` database with matching schemas
(see accompanying `analysis_queries.sql` / table DDL).

In [ ]:
from sqlalchemy import create_engine

# Connection string: postgresql://<username>@<host>:<port>/<database>
# Adjust the username to match your local PostgreSQL role
engine = create_engine("postgresql://yunusajib@localhost:5432/retail_analysis")

df_clean_export.to_sql('clean_sales', engine, if_exists='append', index=False)
df_cancellations_export.to_sql('cancellations', engine, if_exists='append', index=False)

print("Data successfully loaded into PostgreSQL.")
print(f"  clean_sales:    {df_clean_export.shape[0]:,} rows")
print(f"  cancellations:  {df_cancellations_export.shape[0]:,} rows")


## 7. Verify the Load

Run a quick sanity check to confirm the row counts in PostgreSQL match the
cleaned dataframes above (best done directly in DBeaver/psql, shown here for completeness):

```sql
SELECT COUNT(*) FROM clean_sales;       -- expect ~779,425
SELECT COUNT(*) FROM cancellations;     -- expect ~19,494

SELECT invoice_date, total_price
FROM clean_sales
LIMIT 5;                                -- confirm dates and totals loaded correctly, not null
```

With the data verified in PostgreSQL, analysis continues in `analysis_queries.sql`.